#Penjelasan Notebook
Bagian ini akan membahas tahap pembuatan model untuk mengklasifikasi reviu sebuah restoran sebagai positive review atau negative review. Sebelum membuat model, kita perlu mempersiapkan semua library yang dibutuhkan.

In [17]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
import matplotlib.pyplot as plt
import numpy as np
import nltk
from nltk.corpus import stopwords
import json

In [18]:
# menyiapkan dataset
df = pd.read_csv('yelp_labelled.txt', names=['sentence', 'label'], sep='\t')

Kemudian, lakukan tahap preprocessing data berikut:

Mengubah seluruh text kedalam bentuk lowercase.
Menghilangkan stopwords.
Melakukan split dataset.
Melakukan tokenisasi.
Membuat sequences dan melakukan padding.

Selain tahap preprocessing, kita juga perlu menyiapkan metadata dan menyimpannya dalam file word_index.json.

In [19]:
# Mengubah seluruh text kedalam bentuk lowercase
df['sentence'] = df['sentence'].str.lower()

# Menghilangkan stopwords
nltk.download('stopwords')
stop_word = set(stopwords.words('english'))

df['sentence'] = df['sentence'].apply(lambda x:' '.join([word for word in x.split() if word not in (stop_word)]))

# Melakukan split dataset
sentence = df['sentence'].values
label = df['label'].values

sentence_train, sentence_test, label_train, label_test = train_test_split(sentence, label, test_size=0.2, shuffle=False)

# Membuat tokenisasi
filt = '!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ' # Filter untuk menghilangkan symbols

tokenizer = Tokenizer(num_words=2000, oov_token="<OOV>", filters=filt)

tokenizer.fit_on_texts(sentence_train)

# Menyimpan word_index kedalam sebuah file json
word_index = tokenizer.word_index

with open('word_index.json', 'w') as fp:
    json.dump(word_index, fp)

# Membuat sequences dan melakukan padding
train_sekuens = tokenizer.texts_to_sequences(sentence_train)
test_sekuens = tokenizer.texts_to_sequences(sentence_test)

train_padded = pad_sequences(train_sekuens,
                             maxlen=20,
                             padding='post',
                             truncating='post')
test_padded = pad_sequences(test_sekuens,
                            maxlen=20,
                            padding='post',
                            truncating='post')

<>:17: SyntaxWarning: invalid escape sequence '\]'
<>:17: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_527/3114828614.py:17: SyntaxWarning: invalid escape sequence '\]'
  filt = '!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ' # Filter untuk menghilangkan symbols
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Tahap berikutnya adalah membuat dan melatih model menggunakan dataset yang telah kita siapkan sebelumnya.

In [20]:
# Membuat model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(20,)),
    Embedding(2000, 20),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train model
num_epochs = 30
history = model.fit(train_padded, label_train,
                    epochs=num_epochs,
                    validation_data=(test_padded, label_test),
                    verbose=1)

Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5638 - loss: 0.6882 - val_accuracy: 0.2400 - val_loss: 0.7490
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5650 - loss: 0.6809 - val_accuracy: 0.2400 - val_loss: 0.7805
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5650 - loss: 0.6756 - val_accuracy: 0.2400 - val_loss: 0.7905
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5725 - loss: 0.6613 - val_accuracy: 0.2400 - val_loss: 0.7563
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6850 - loss: 0.6168 - val_accuracy: 0.2950 - val_loss: 0.7918
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7700 - loss: 0.5325 - val_accuracy: 0.3600 - val_loss: 0.8054
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8775 - loss: 0.4046 - val_accuracy: 0.8100 - val_loss: 0.5184
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9312 - loss: 0.2794 - val_accuracy: 0.7350 - val_loss

In [21]:
# Jika performa model dirasa cukup, simpanlah dalam format HDF5 menggunakan perintah berikut:
model.save("model.h5")

In [22]:
#Nah, sebelum men-deploy model ke dalam web, ubahlah HDF5 model yang telah disimpan tadi ke dalam bentuk file json.

# Install tensorflowjs
!pip install tensorflowjs

# Convert model.h5 to model
!tensorflowjs_converter --input_format=keras model.h5 tfjs_model

2026-04-29 02:39:53.135825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777430393.158319    3759 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777430393.164930    3759 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777430393.182551    3759 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777430393.182595    3759 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777430393.182599    3759 computation_placer.cc:177] computation placer alr

Jika semua tahapan berjalan dengan baik, akan muncul sebuah folder baru bernama `tfjs_model` yang berisi `model.json` & weight berbentuk binary file. Selamat! Model Anda telah siap untuk masuk ke tahap deployment dengan TensorFlow.js.

#Menerapkan Model pada Web Browser dengan TensorFlow.js
Untuk menerapkan model yang telah dibuat di web browser, buatlah file ‘index.html’ sebagai tampilan utama web. Selain itu, buatlah file ‘script.js’ yang berisi perintah untuk men-deploy model.